In [25]:
import duckdb
import pandas as pd
import numpy as np  
import plotly.express as px

In [ ]:
DB_PATH = 'project_data.duckdb'
con = duckdb.connect(database=DB_PATH, read_only=True)

In [ ]:
query = """
    SELECT * 
    FROM serving.fct_desigualdade_gestao_esforco
    WHERE tx_esforco_docente_critico IS NOT NULL
"""
df = con.execute(query).df()
con.close()

In [ ]:
limites = [-np.inf, 0, 25, 50, np.inf]
rotulos = ['0% (Sem Risco)', '1% a 25% (Atenção)', '26% a 50% (Alto Risco)', '> 50% (Crítico)']

df['faixa_risco_esforco'] = pd.cut(
    df['tx_esforco_docente_critico'], 
    bins=limites, 
    labels=rotulos,
    right=True
)

# Calcula a quantidade de escolas por Rede, Raça e Faixa de Risco
df_agrupado = df.groupby(['no_dependencia', 'perfil_raca', 'faixa_risco_esforco'], observed=True).size().reset_index(name='qtd_escolas')

# Remove categorias vazias caso alguma rede não tenha escolas com maioria negra, por exemplo
df_agrupado = df_agrupado[df_agrupado['qtd_escolas'] > 0]

# Calcula o % (Proporção) para normalizar a barra em 100%
df_agrupado['pct_escolas'] = df_agrupado['qtd_escolas'] / df_agrupado.groupby(['no_dependencia', 'perfil_raca'])['qtd_escolas'].transform('sum')

fig1 = px.bar(
    df_agrupado,
    x='perfil_raca',
    y='pct_escolas',
    color='faixa_risco_esforco',
    facet_col='no_dependencia',
    title='Alocação de Gestores nas Faixas de Risco de Esforço Docente',
    labels={
        'perfil_raca': '',
        'pct_escolas': 'Proporção de Escolas (%)',
        'faixa_risco_esforco': 'Risco de Sobrecarga',
        'no_dependencia': 'Rede'
    },
    template='plotly_white',
    
    color_discrete_map={
        '0% (Sem Risco)': '#457b9d', 
        '1% a 25% (Atenção)': '#a8dadc', 
        '26% a 50% (Alto Risco)': '#e63946', 
        '> 50% (Crítico)': '#780000'
    }
)

fig1.update_layout(
    title_font_size=20,
    legend_title_font_weight='bold',
    yaxis_tickformat='.0%' 
)


fig1.update_traces(hovertemplate='<b>%{x}</b><br>Proporção: %{y:.1%}<br>Total Escolas: %{customdata[0]}',
                   customdata=df_agrupado[['qtd_escolas']])


fig1.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig1.show()

In [ ]:
fig2 = px.histogram(
    df, 
    x='nivel_complexidade', 
    color='perfil_genero', 
    barnorm='fraction',
    title='Proporção de Gênero da Gestão x Nível de Complexidade',
    labels={
        'nivel_complexidade': 'Nível de Complexidade da Gestão (ICG)',
        'perfil_genero': 'Perfil de Gênero (Gestão)'
    },
    template='plotly_white',
    color_discrete_map={
        'Maioria Feminina': '#A2D2FF',
        'Maioria Masculina': '#FFAFCC',
        'Equilibrado / Não Informado': '#CDB4DB'
    }
)

# Ajustes de design para o Histograma
fig2.update_layout(
    yaxis_title="Proporção (0 a 1)",
    title_font_size=20,
    legend_title_font_weight='bold',
    xaxis={'categoryorder': 'category ascending'}
)

# Modifica o texto flutuante para mostrar porcentagem
fig2.update_traces(hovertemplate='%{y:.1%}')

# Exibe o segundo gráfico
fig2.show()